# WaferGuard FAB 로컬 데이터 분석

Google Drive나 클라우드 저장소를 사용하지 않습니다. 원본은 `data/input`, 결과는 `data/output`에 저장됩니다. 이 notebook은 셀 단위 학습용 UI이고, 실제 로직은 `app/services/fab_analysis.py`에 있으므로 설치형 프로그램에서는 `scripts/run_fab_analysis.py`로 동일하게 실행할 수 있습니다.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import JSON, display

from app.services.fab_analysis import (
    DATA_INPUT_DIR, DATA_OUTPUT_DIR, build_candidate_bundle, dataset_profile,
    export_runtime_features, feature_importance_analysis, load_local_dataset,
    save_analysis_outputs,
)

sns.set_theme(style='whitegrid')
print('Project root:', ROOT)
print('Training data:', DATA_INPUT_DIR)
print('Analysis output:', DATA_OUTPUT_DIR)

## 1. 설정

- 직접 준비한 CSV/Parquet는 `DATA_FILE` 위치에 둡니다.
- 현재 로컬 DB의 실제 runtime feature를 쓰려면 `EXPORT_FROM_LOCAL_DB=True`로 바꿉니다.
- `CREATE_CANDIDATE=False`가 기본이므로 EDA만으로 모델 artifact가 만들어지지 않습니다.

In [ ]:
PROCESS_ID = 'cmp'  # photo | etch | deposition | cmp
DATA_FILE = DATA_INPUT_DIR / 'fab_training.csv'
TARGET_COLUMN = 'is_anomaly'  # label이 없으면 None
FEATURE_COLUMNS = None  # None이면 numeric feature 자동 선택
EXPORT_FROM_LOCAL_DB = False
CREATE_CANDIDATE = False
CONTAMINATION = 0.05
RANDOM_STATE = 42

## 2. 로컬 데이터 로드

DB export는 `process_telemetry.detector_context`의 실제 feature vector와 detector label을 CSV로 평탄화합니다. 직접 넣은 파일도 EDA는 가능하지만 fingerprint metadata가 없으면 Production 호환 후보 저장은 차단됩니다.

In [ ]:
if EXPORT_FROM_LOCAL_DB:
    DATA_FILE = export_runtime_features(PROCESS_ID, DATA_INPUT_DIR / f'{PROCESS_ID}_runtime_features.csv')

df = load_local_dataset(DATA_FILE)
print(f'Loaded {len(df):,} rows × {len(df.columns):,} columns from {DATA_FILE}')
display(df.head())

## 3. 데이터 구조, 타입, 결측, 중복

In [ ]:
profile = dataset_profile(df, TARGET_COLUMN)
display(JSON(profile))

column_structure = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'non_null': df.notna().sum(),
    'missing': df.isna().sum(),
    'missing_ratio': df.isna().mean(),
    'unique': df.nunique(dropna=False),
}).sort_values(['missing', 'unique'], ascending=[False, False])
display(column_structure)
if profile['missing_cells']:
    plt.figure(figsize=(12, 4))
    sns.heatmap(df.head(500).isna(), cbar=False, yticklabels=False)
    plt.title('Missing-value map (first 500 rows)')
    plt.tight_layout()
    plt.show()

In [ ]:
duplicate_rows = df[df.duplicated(keep=False)]
print('Duplicate rows:', len(duplicate_rows))
display(duplicate_rows.head(20))

## 4. 기술 통계와 target 분포

In [ ]:
display(df.describe(include='all').T)
numeric_preview = [name for name in df.select_dtypes(include='number').columns if name != TARGET_COLUMN][:12]
if numeric_preview:
    df[numeric_preview].hist(figsize=(14, 9), bins=25)
    plt.suptitle('Numeric feature distributions')
    plt.tight_layout()
    plt.show()
if TARGET_COLUMN and TARGET_COLUMN in df:
    target_counts = df[TARGET_COLUMN].value_counts(dropna=False)
    display(target_counts.to_frame('count'))
    target_counts.plot(kind='bar', title=f'Target distribution: {TARGET_COLUMN}', figsize=(7, 3))
    plt.tight_layout()
    plt.show()

## 5. 공통 분석 실행

label이 있으면 supervised Random Forest, 없으면 Isolation Forest pseudo-label을 설명하는 surrogate Random Forest를 사용합니다. 후자의 importance는 ground truth 설명이 아니라는 점이 결과에 명시됩니다.

In [ ]:
analysis = feature_importance_analysis(
    df, target_column=TARGET_COLUMN, feature_columns=FEATURE_COLUMNS, random_state=RANDOM_STATE
)
print('Selected features:', len(analysis['feature_columns']))
display(JSON(analysis['metrics']))
if analysis['metrics'].get('confusion_matrix'):
    labels = analysis['metrics']['labels']
    matrix = pd.DataFrame(analysis['metrics']['confusion_matrix'], index=labels, columns=labels)
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion matrix')
    plt.tight_layout()
    plt.show()

## 6. Correlation heatmap

In [ ]:
correlation = analysis['correlation']
size = max(8, min(24, len(correlation.columns) * 0.55))
plt.figure(figsize=(size, size))
sns.heatmap(correlation, cmap='coolwarm', center=0, vmin=-1, vmax=1, square=True)
plt.title('FAB Feature Correlation')
plt.tight_layout()
plt.show()
display(pd.DataFrame(analysis['top_correlations']))

## 7. Feature importance

In [ ]:
importance = analysis['feature_importance'].head(20).sort_values('importance')
display(importance.sort_values('importance', ascending=False))
importance.plot.barh(x='feature', y='importance', legend=False, figsize=(9, 7), title='Top Feature Importance')
plt.tight_layout()
plt.show()

## 8. 결과 저장과 선택적 candidate 생성

`CREATE_CANDIDATE=True`는 runtime export 데이터처럼 contract/fingerprint가 정확할 때만 동작합니다. artifact는 Staging 등록 후보일 뿐 자동 승격되지 않습니다.

In [ ]:
candidate_bundle = None
registration_metrics = None
if CREATE_CANDIDATE:
    candidate_bundle, registration_metrics = build_candidate_bundle(
        df, process_id=PROCESS_ID, feature_columns=analysis['feature_columns'],
        target_column=TARGET_COLUMN, contamination=CONTAMINATION, random_state=RANDOM_STATE,
    )

summary = save_analysis_outputs(
    df, analysis, source_path=DATA_FILE, process_id=PROCESS_ID, target_column=TARGET_COLUMN,
    output_dir=DATA_OUTPUT_DIR, candidate_bundle=candidate_bundle,
    registration_metrics=registration_metrics,
)
display(JSON(summary))

## 9. 대시보드 반영

저장된 `data/output/dashboard_summary.json`은 backend의 `GET /api/v1/fab/analysis/latest`에서 읽습니다. Backend와 frontend를 실행한 뒤 **AI Analysis → Local Data Analysis**에서 데이터 크기, 결측/중복, 분석 방식, 상위 importance와 correlation을 확인합니다. 원본 CSV/Parquet는 API로 노출되지 않습니다.